In [1]:
import torch

torch.cuda.is_available()

True

In [3]:
print(torch.cuda.get_device_name())
free_mem, total_mem = torch.cuda.mem_get_info()
print(f"Free VRAM: {free_mem / 1024**3:.2f} GB")
print(f"Total VRAM: {total_mem / 1024**3:.2f} GB")

NVIDIA GeForce RTX 3060
Free VRAM: 10.30 GB
Total VRAM: 11.63 GB


# Tensors

In [3]:
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])
tensor3d = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
tensor1p5d = torch.tensor([1])

In [4]:
print(tensor1d.dtype)
print(tensor1d.to(torch.float32).dtype)


torch.int64
torch.float32


In [5]:
print(tensor0d.shape)
print(tensor3d.shape)
print(tensor1p5d.shape)
print(tensor2d.shape)
print(tensor2d.reshape(3, 2))
print(tensor2d.T)
print(tensor2d @ tensor2d.T)

torch.Size([])
torch.Size([2, 2, 2])
torch.Size([1])
torch.Size([2, 3])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[1, 4],
        [2, 5],
        [3, 6]])
tensor([[14, 32],
        [32, 77]])


# Computation Graphs

In [6]:
import torch.nn.functional as F

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])
b = torch.tensor([0.0])

z = w1 * x1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)
print(loss)

tensor(0.0852)


# Autograd

In [7]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [8]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


# Multilayer neural networks

In [9]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [10]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [11]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [12]:
print(model.layers[0].weight)
print(model.layers[0].weight.shape)
print(model.layers[0].bias)
print(model.layers[0].bias.shape)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)
torch.Size([30, 50])
Parameter containing:
tensor([-0.1250,  0.0513,  0.0366,  0.0075,  0.0509,  0.0545, -0.0393,  0.0924,
        -0.1412, -0.1232, -0.1063,  0.0081, -0.1249,  0.0101, -0.0019, -0.1298,
         0.1388, -0.0330,  0.1017,  0.1247, -0.0554, -0.0417,  0.1388,  0.0159,
         0.1215,  0.0385,  0.0769, -0.1224, -0.0279,  0.0991],
       requires_grad=True)
torch.Size([30])


In [13]:
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [14]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])


In [15]:
out.sum()

tensor(1.)

# Data loader

In [16]:
X_train = torch.tensor([[-1.2, 3.1], [-0.9, 2.9], [-0.5, 2.6], [2.3, -1.1], [2.7, -1.5]])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor(
    [
        [-0.8, 2.8],
        [2.6, -1.6],
    ]
)
y_test = torch.tensor([0, 1])

In [17]:
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]


train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [18]:
print(len(train_ds))

5


In [19]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(dataset=train_ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True)

test_loader = DataLoader(dataset=test_ds, batch_size=2, shuffle=True, num_workers=0)

In [20]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx + 1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


# Training Loop

In [21]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch + 1:03d}/{num_epochs:03d} | Batch {batch_idx:03d}/{len(train_loader):03d} | Train Loss: {loss:.2f}")

    model.eval()

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00


In [22]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [23]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


In [24]:
predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [25]:
def compute_accuracy(model, dataloader):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [26]:
print(compute_accuracy(model, train_loader))
print(compute_accuracy(model, test_loader))

1.0
1.0


In [27]:
torch.save(model.state_dict(), "appendix_a_model.pth")

In [28]:
model.state_dict()

OrderedDict([('layers.0.weight',
              tensor([[-0.3091,  0.1047],
                      [-0.3552,  0.2827],
                      [-0.6039,  0.5246],
                      [-0.5140, -0.5622],
                      [-0.4623,  0.3811],
                      [-0.2791,  0.3349],
                      [-0.6001, -0.4290],
                      [-0.2596, -0.1390],
                      [-0.5323,  0.4353],
                      [-0.1786,  0.2737],
                      [ 0.5025,  0.1076],
                      [ 0.1365,  0.7400],
                      [-0.3292,  0.2645],
                      [-0.3306,  0.5677],
                      [ 0.6177, -0.6788],
                      [ 0.5121,  0.3864],
                      [ 0.2444,  0.3179],
                      [ 0.6392, -0.1203],
                      [-0.5988, -0.2008],
                      [-0.5104,  0.0951],
                      [-0.0307, -0.4296],
                      [-0.0750,  0.7119],
                      [ 0.0188, -0.0559],
 

In [29]:
model = NeuralNetwork(2, 2)
model.load_state_dict(torch.load("appendix_a_model.pth", weights_only=True))

<All keys matched successfully>

# GPU

In [30]:
tensor_1 = torch.tensor([1.0, 2.0, 3.0])
tensor_2 = torch.tensor([4.0, 5.0, 6.0])
print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


In [31]:
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")
print(tensor_1 + tensor_2)


tensor([5., 7., 9.], device='cuda:0')


In [32]:
print(tensor_1.to("cpu") + tensor_2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [ ]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device=device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"{device} Epoch: {epoch + 1:03d}/{num_epochs:03d} | Batch {batch_idx:03d}/{len(train_loader):03d} | Train Loss: {loss:.2f}")

    model.eval()

cuda Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
cuda Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
cuda Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
cuda Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
cuda Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
cuda Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00
